# <ins>Retail Data Analytics</ins>

**Background**
This project will analyse retail data. The aims are first to load, clean, and tranform the data. Second, to extract meaningful information about sales around prominent holidays. Third, to display the insights in easily understandable and actionable ways.

**Data**
Data comes from 45 stores, located in different regions, each with multiple departments. Granularity is weeks, of which there are special ones designated as <ins>*markdown periods*</ins> around prominent holidays, which are weighted <ins>*5x higher*</ins> than normal ones.

3 datasets
1. Stores
2. Features
    - Store - the store number
    - Date - the week
    - Temperature - average temperature in the region
    - Fuel_Price - cost of fuel in the region
    - MarkDown1-5 - anonymized data related to promotional markdowns. <ins>*MarkDown data is only available after Nov 2011, and is not available for all stores all the time. Any missing value is marked with an NA*</ins>
    - CPI - the consumer price index
    - Unemployment - the unemployment rate
    - IsHoliday - whether the week is a special holiday week
3. Sales:
    - Store - the store number
    - Dept - the department number
    - Date - the week
    - Weekly_Sales -  sales for the given department in the given store
    - IsHoliday - whether the week is a special holiday week


**Project Layout**
1. Preliminary data exploration
2. Data cleaning
3. Data transformation and feature engineering
4. Exploratory data analysis + visualisations



In [1]:
# import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px


# 1. Preliminary data exploration

## Load and Combine

In [2]:
# load data
stores = pd.read_csv('stores data-set.csv')
sales = pd.read_csv('sales data-set.csv')
features = pd.read_csv('Features data set.csv')

In [3]:
stores.head()

,Store,Type,Size
0,1,A,151315
1,2,A,202307
2,3,B,37392
3,4,A,205863
4,5,B,34875


In [4]:
stores.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 45 entries, 0 to 44
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Store   45 non-null     int64 
 1   Type    45 non-null     object
 2   Size    45 non-null     int64 
dtypes: int64(2), object(1)
memory usage: 1.2+ KB


In [5]:
sales.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday
0,1,1,05/02/2010,24924.50,False
1,1,1,12/02/2010,46039.49,True
2,1,1,19/02/2010,41595.55,False
3,1,1,26/02/2010,19403.54,False
4,1,1,05/03/2010,21827.90,False


In [6]:
sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 421570 entries, 0 to 421569
Data columns (total 5 columns):
 #   Column        Non-Null Count   Dtype  
---  ------        --------------   -----  
 0   Store         421570 non-null  int64  
 1   Dept          421570 non-null  int64  
 2   Date          421570 non-null  object 
 3   Weekly_Sales  421570 non-null  float64
 4   IsHoliday     421570 non-null  bool   
dtypes: bool(1), float64(1), int64(2), object(1)
memory usage: 13.3+ MB


In [7]:
features.head()

,Store,Date,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday
0,1,05/02/2010,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False
1,1,12/02/2010,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True
2,1,19/02/2010,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False
3,1,26/02/2010,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False
4,1,05/03/2010,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False


In [8]:
features.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8190 entries, 0 to 8189
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   Store         8190 non-null   int64  
 1   Date          8190 non-null   object 
 2   Temperature   8190 non-null   float64
 3   Fuel_Price    8190 non-null   float64
 4   MarkDown1     4032 non-null   float64
 5   MarkDown2     2921 non-null   float64
 6   MarkDown3     3613 non-null   float64
 7   MarkDown4     3464 non-null   float64
 8   MarkDown5     4050 non-null   float64
 9   CPI           7605 non-null   float64
 10  Unemployment  7605 non-null   float64
 11  IsHoliday     8190 non-null   bool   
dtypes: bool(1), float64(9), int64(1), object(1)
memory usage: 712.0+ KB


In [27]:
# work out the 'hierarchy' of the dataframes based on their granularity
print(f"Stores have unique store numbers? - {stores['Store'].nunique() == stores.shape[0]}")
print(f"Features has unique store numbers - date combinations? - {features[['Store', 'Date']].duplicated().sum() == 0}")
print(f"Sales has unique store numbers - department - date combinations? - {sales[['Store','Dept', 'Date']].duplicated().sum() == 0}")

Stores have unique store numbers? - True
Features has unique store numbers - date combinations? - True
Sales has unique store numbers - department - date combinations? - True


The above suggests that <ins>Sales</ins> has the highest granularity (unique values for each store + department + date), followed by Features and Stores => Sales is the main dataframe, Features will be merged into it, followed by Stores at the end.

In [13]:
# merge the dataframes
df = sales.merge(features, how='left', on=['Store', 'Date']).merge(stores, how='left', on='Store')
df.head()

,Store,Dept,Date,Weekly_Sales,IsHoliday_x,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,IsHoliday_y,Type,Size
0,1,1,05/02/2010,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,False,A,151315
1,1,1,12/02/2010,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,True,A,151315
2,1,1,19/02/2010,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,False,A,151315
3,1,1,26/02/2010,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,False,A,151315
4,1,1,05/03/2010,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,False,A,151315


In [21]:
# col 'IsHoliday' appeared in both Sales and Features, it can be used to check the quality of the merge
# values not matching would suggest a merge issue
(df['IsHoliday_x'] == df['IsHoliday_y']).to_frame().iloc[:].value_counts()


0   
True    421570
Name: count, dtype: int64

In [19]:
df.shape

(421570, 17)

All values are matching! Whilst this does **NOT** guarantee the lack of issues, it does at least disprove that there are any obvious mismatches.

## Quality check

# 2. Data cleaning

# 3. Data transformation and feature engineering

# 4. Exploratory data analysis + visualisations